In [5]:
import numpy as np
from scipy.stats import norm
import math
import random
from datetime import date, timedelta
from typing import Dict, List, Tuple


# =========================
# BLACK-SCHOLES SECTION
# =========================
def d1(S, K, T, r, sigma, q=0.0):
    return (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

def d2(S, K, T, r, sigma, q=0.0):
    return d1(S, K, T, r, sigma, q) - sigma * np.sqrt(T)


def bs_price(S, K, T, r, sigma, option_type="call", q=0.0):
    D1 = d1(S, K, T, r, sigma, q)
    D2 = d2(S, K, T, r, sigma, q)

    if option_type == "call":
        return S * np.exp(-q*T) * norm.cdf(D1) - K * np.exp(-r*T) * norm.cdf(D2)
    elif option_type == "put":
        return K * np.exp(-r*T) * norm.cdf(-D2) - S * np.exp(-q*T) * norm.cdf(-D1)
    else:
        raise ValueError("option_type must be 'call' or 'put'")


# =========================
# ZERO COUPON BOND
# =========================
def zero_coupon_bond_price(face_value, rate, time):
    return face_value / (1 + rate) ** time


# =========================
# ELN PAYOFF + MC
# =========================
def payoff(paths: Dict[date, List[float]]) -> List[Tuple[date, float]]:
    
    maturity_date = date(2026, 3, 23)
    initial_price = 314.619
    barrier_price = 226.5257
    cap_price = 320.9114
    coupon = 0.1585 * 0.25

    final_determination_date = date(2026, 3, 19)
    final_price = paths[final_determination_date][0]

    cashflows = []

    # Coupons
    cashflows.append((date(2025, 12, 23), coupon * initial_price))
    cashflows.append((date(2026, 3, 19), coupon * initial_price))

    # Dividends
    cashflows.append((date(2025, 12, 22), 1.71))
    cashflows.append((date(2026, 3, 19), 1.80))

    # Barrier check
    barrier_event = False
    for prices in paths.values():
        if min(prices) <= barrier_price:
            barrier_event = True
            break

    capped_final = min(final_price, cap_price)

    if barrier_event:
        maturity_payoff = capped_final
    else:
        maturity_payoff = max(initial_price, capped_final)

    cashflows.append((maturity_date, maturity_payoff))

    return cashflows


def generate_path(S0, r, sigma, start_date, end_date):
    paths = {}
    current_price = S0
    current_date = start_date

    while current_date <= end_date:
        paths[current_date] = [current_price]

        dt = 1/252
        z = random.gauss(0, 1)
        current_price *= math.exp((r - 0.5 * sigma**2) * dt + sigma * math.sqrt(dt) * z)

        current_date += timedelta(days=1)

    return paths


def discount_cashflows(cashflows, r, valuation_date):
    pv = 0.0
    for d, cf in cashflows:
        t = (d - valuation_date).days / 365
        pv += cf * math.exp(-r * t)
    return pv


def price_eln_mc(S0, r, sigma, n_sim=5000):
    valuation_date = date(2026, 2, 3)
    maturity_date = date(2026, 3, 23)

    total_value = 0.0

    for _ in range(n_sim):
        path = generate_path(S0, r, sigma, valuation_date, maturity_date)
        cfs = payoff(path)
        total_value += discount_cashflows(cfs, r, valuation_date)

    return total_value / n_sim


# =========================
# UNIFIED PRICER
# =========================
class MultiAssetPricer:

    def price(
        self,
        product_type,
        asset_type=None,
        S=100,
        K=100,
        T=1,
        r=0.05,
        sigma=0.2,
        option_type="call",
        **kwargs
    ):
        """
        product_type:
            'option' → Black-Scholes
            'eln'    → Monte Carlo ELN
            'bond'   → Zero Coupon Bond
        """

        # -------- OPTION --------
        if product_type == "option":

            if asset_type == "fx":
                q = kwargs.get("foreign_rate", 0.0)

            elif asset_type == "commodity":
                q = kwargs.get("convenience_yield", 0.0)

            else:  # equity/index
                q = kwargs.get("dividend_yield", 0.0)

            return bs_price(S, K, T, r, sigma, option_type, q)

        # -------- ELN --------
        elif product_type == "eln":
            return price_eln_mc(
                S0=S,
                r=r,
                sigma=sigma,
                n_sim=kwargs.get("n_sim", 5000)
            )

        # -------- BOND --------
        elif product_type == "bond":
            face_value = kwargs.get("face_value", 1000)
            return zero_coupon_bond_price(face_value, r, T)

        else:
            raise ValueError("Unsupported product_type")


# =========================
# USAGE
# =========================
if __name__ == "__main__":

    pricer = MultiAssetPricer()

    # ---- Option ----
    option_price = pricer.price(
        product_type="option",
        asset_type="index",
        S=100,
        K=100,
        T=1,
        r=0.05,
        sigma=0.2,
        option_type="call",
        dividend_yield=0.02
    )
    print("Option Price:", option_price)

    # ---- ELN ----
    eln_price = pricer.price(
        product_type="eln",
        S=314.619,
        r=0.035,
        sigma=0.32,
        n_sim=3000
    )
    print("ELN Price:", eln_price)

    # ---- Zero Coupon Bond ----
    bond_price = pricer.price(
        product_type="bond",
        face_value=1000,
        r=0.06,
        T=5
    )
    print("Zero Coupon Bond Price:", round(bond_price, 2))

Option Price: 9.227005508154036
ELN Price: 343.4909958526605
Zero Coupon Bond Price: 747.26
